## Imports

In [1]:
import os
import re
import glob
import time

import numpy as np
import pandas as pd
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 50)

In [3]:
## Path

cwd = os.getcwd()
print(cwd)
final = '/Users/E2074/local-datasets/customer/loyalty/experiment1/final'
experiment_refresh = '/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh'

print(final)
print(experiment_refresh)

/Users/E2074/analytics/customer/loyalty/experiment1
/Users/E2074/local-datasets/customer/loyalty/experiment1/final
/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh


## Exclude from First Ride Selector 

### Clipboard the >=1 orderstatus-dropped customer numbers.

In [9]:
def fetch_net_rides():
    
    df_first_ride = pd.read_clipboard()
    df_first_ride['Phone'] = df_first_ride['Phone'].astype(str).str[-10:]
    print(df_first_ride.shape)
    display(df_first_ride.head(2))
    
    df_first_ride.to_parquet(experiment_refresh + '/exclude_from_first_ride/net_customers_20250417.parquet', index=False)
    
    return df_first_ride

df_first_ride = fetch_net_rides()

(25611, 1)


,Phone
0,9611413822
1,9880708001


In [10]:
df_first_ride[df_first_ride['Phone'].isin(['7639345071', '9785494112', '8108282883', '7983804744', '9845095045'])]

,Phone


### Generate updated exclude csv file for User-Selector

In [11]:
def generate_ct_first_ride_exclude():
    path = '/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh/exclude_from_first_ride/'
    
    parquet_files = glob.glob(os.path.join(path, "*.parquet"))
    print(parquet_files)
    df = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)
    df.drop_duplicates(inplace=True)
    print(df.shape)
    display(df.head(2))

    
    df.to_csv(path + 'second_ride/second_ride_movement/first_ride_exclude_list.csv', index=False)
    df.to_csv(path + 'ct_exclude_file/first_ride_exclude_list.csv',  header=False, index=False)
    
    return df

df_excluded_group = generate_ct_first_ride_exclude()

['/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh/exclude_from_first_ride/net_customers_20250412.parquet', '/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh/exclude_from_first_ride/net_customers_20250413.parquet', '/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh/exclude_from_first_ride/net_customers_20250415.parquet', '/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh/exclude_from_first_ride/net_customers_20250414.parquet', '/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh/exclude_from_first_ride/net_customers_20250416.parquet', '/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh/exclude_from_first_ride/net_customers_20250417.parquet']
(131029, 1)


,Phone
0,9538398310
1,9901690919


In [12]:
df_excluded_group.head()

,Phone
0,9538398310
1,9901690919
2,9066242536
3,9720011016
4,8004488782


In [13]:
df_excluded_group.shape

(131029, 1)

## Moved to second ride

### Clipboard the = 1 orderstatus-dropped customer numbers.

In [14]:
def fetch_net_rides():
    
    df = pd.read_clipboard()
    df['Phone'] = df['Phone'].astype(str).str[-10:]
    print(df.shape)
    display(df.head(2))
    
    df.to_parquet(experiment_refresh + '/exclude_from_first_ride/second_ride/next_ride_customers_20250417.parquet', index=False)
    
    return df

df_second_ride = fetch_net_rides()

(25611, 1)


,Phone
0,9611413822
1,9880708001


### Clipboard the >= 3 orderstatus-dropped customer numbers. (Experiment date)

In [15]:
def exited_customers():
    
    df = pd.read_clipboard()
    df['Phone'] = df['Phone'].astype(str).str[-10:]
    print(df.shape)
    display(df.head(2))
    
    df.to_parquet(experiment_refresh + '/exclude_from_first_ride/second_ride/ct_exit_file/more_than_2_rides.parquet', index=False)
    
    return df

df_exited_customer = exited_customers()

(25611, 1)


,Phone
0,9611413822
1,9880708001


In [16]:
df_exited_customer = pd.read_parquet(experiment_refresh + '/exclude_from_first_ride/second_ride/ct_exit_file/more_than_2_rides.parquet')
df_exited_customer

,Phone
0,9611413822
1,9880708001
2,9824629298
3,9740414851
4,9916480799
...,...
25606,6377967233
25607,9972940686
25608,7892562163
25609,9113832425


In [17]:
new_row = pd.DataFrame([{'Phone': '8778433813'}])
df_exited_customer = pd.concat([df_exited_customer, new_row], ignore_index=True)

In [18]:
df_exited_customer = df_exited_customer[df_exited_customer['Phone'] == '8778433813']

In [19]:
df_exited_customer

,Phone
25611,8778433813


### Generate updated second ride csv file for User-Selector

In [20]:
def generate_ct_second_ride_include():
    path = '/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh/exclude_from_first_ride/second_ride/'
    
    parquet_files = glob.glob(os.path.join(path, "*.parquet"))
    print(parquet_files)
    df = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)


    df_filtered = df[~df['Phone'].isin(df_exited_customer['Phone'])]
    df_filtered.drop_duplicates(inplace=True)
    print(df_filtered.shape)
    display(df_filtered.head(2))
    
    df_filtered.to_csv(path + 'ct_include_file/second_ride_include_list.csv',  header=False, index=False)
    
    return df_filtered

df_include_group = generate_ct_second_ride_include()

['/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh/exclude_from_first_ride/second_ride/next_ride_customers_20250412.parquet', '/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh/exclude_from_first_ride/second_ride/next_ride_customers_20250413.parquet', '/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh/exclude_from_first_ride/second_ride/next_ride_customers_20250416.parquet', '/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh/exclude_from_first_ride/second_ride/next_ride_customers_20250417.parquet', '/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh/exclude_from_first_ride/second_ride/next_ride_customers_20250415.parquet', '/Users/E2074/local-datasets/customer/loyalty/experiment1/final/experiment_refresh/exclude_from_first_ride/second_ride/next_ride_customers_20250414.parquet']
(131029, 1)


,Phone
0,9538398310
1,9901690919


In [21]:
df_include_group

,Phone
0,9538398310
1,9901690919
2,9066242536
3,9720011016
4,8004488782
...,...
152284,6360798811
152285,6282368136
152286,7338659998
152287,8861264463
